# Imputación de faltantes y valores atípicos de DViento

Este notebook realiza la imputación de **faltantes** y **valores atípicos** de **DViento** tratando la variable como **circular**.

La dirección del viento no se modela directamente como número lineal. Primero se transforma en:

- **seno(ángulo)**
- **coseno(ángulo)**

Luego, después de imputar, se reconstruye el ángulo final en grados.

## Mejora incorporada
Además de detectar fechas faltantes, el notebook identifica valores atípicos:

1. Valores fuera del rango angular válido `[0, 360)`.
2. Valores con desviación angular anormal frente al comportamiento local de la serie.

Los atípicos se conservan como trazabilidad, se convierten temporalmente en faltantes y se imputan con Random Forest circular.

**Actualización:** se agregó medición del tiempo total de ejecución del notebook y exportación del archivo CSV correspondiente.

**Actualización:** se agregó gráfica comparativa antes/después de la imputación, con marcadores para faltantes y atípicos corregidos.

**Actualización:** las gráficas antes/después ahora se generan automáticamente para todas las estaciones disponibles, sin escribir manualmente el nombre de la estación.

**Corrección técnica:** se agregó la importación `from IPython.display import display, Markdown` para evitar el error `NameError: name 'Markdown' is not defined`.

**Nota:** en `DViento` no se marca el valor 0 como atípico porque 0° es una dirección del viento válida.

**Aclaración de periodo:** el notebook no está configurado para imputar dos años de forma fija. El periodo de imputación se toma automáticamente desde la fecha mínima hasta la fecha máxima del archivo cargado. Si el archivo contiene solo 2022, se imputa únicamente ese año.

In [ ]:
# ============================================================
# INICIO DE MEDICIÓN DEL TIEMPO TOTAL DEL NOTEBOOK
# ============================================================

import time
from datetime import datetime

TIEMPO_INICIO_NOTEBOOK = time.perf_counter()
FECHA_HORA_INICIO_NOTEBOOK = datetime.now()

print("Inicio de ejecución del notebook:")
print(FECHA_HORA_INICIO_NOTEBOOK.strftime("%Y-%m-%d %H:%M:%S"))

In [3]:
import pandas as pd
import numpy as np
import math
import warnings
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")

VARIABLE = "DViento"
RUTA = "DViento.csv"
ARCHIVO_BASE = "DViento"

# Parámetros de detección de atípicos circulares
VENTANA_ANGULAR_ATIPICOS = 15
UMBRAL_DESVIACION_ANGULAR = 90  # grados
MULTIPLICADOR_IQR_ANGULAR = 1.5

print("Configuración cargada para imputación circular con detección de atípicos.")

In [4]:
df = pd.read_csv(RUTA)
df["Fecha inicial"] = pd.to_datetime(df["Fecha inicial"], errors="coerce")
df[VARIABLE] = pd.to_numeric(df[VARIABLE], errors="coerce")
df = df[["Estacion", "Fecha inicial", VARIABLE]].copy()
df = df.sort_values(["Estacion", "Fecha inicial"]).reset_index(drop=True)

print("Registros:", len(df))
print("Columnas:", df.columns.tolist())
print("Estaciones:", sorted(df["Estacion"].dropna().unique()))
print("Rango temporal:", df["Fecha inicial"].min().date(), "a", df["Fecha inicial"].max().date())
display(df.head())

Registros: 1462
Columnas: ['Estacion', 'Fecha inicial', 'DViento']
Estaciones: ['CENTRO DE ALTO RENDIMIENTO', 'SAN CRISTOBAL']
Rango temporal: 2020-01-01 a 2021-12-31


,Estacion,Fecha inicial,DViento
0,CENTRO DE ALTO RENDIMIENTO,2021-12-31,258.695652
1,CENTRO DE ALTO RENDIMIENTO,2021-12-30,223.833333
2,CENTRO DE ALTO RENDIMIENTO,2021-12-29,274.416667
3,CENTRO DE ALTO RENDIMIENTO,2021-12-28,191.041667
4,CENTRO DE ALTO RENDIMIENTO,2021-12-27,188.208333


In [ ]:
# ============================================================
# PERIODO DE IMPUTACIÓN DETECTADO AUTOMÁTICAMENTE
# ============================================================

# El notebook no fuerza dos años. Imputa únicamente el rango de fechas
# contenido en el archivo cargado.
PERIODO_INICIO = df["Fecha inicial"].min()
PERIODO_FIN = df["Fecha inicial"].max()
DIAS_PERIODO = pd.date_range(PERIODO_INICIO, PERIODO_FIN, freq="D").size

print("Periodo detectado en el archivo:")
print("Inicio:", PERIODO_INICIO.date())
print("Fin:", PERIODO_FIN.date())
print("Días esperados:", DIAS_PERIODO)

if DIAS_PERIODO <= 366:
    print("Interpretación: el archivo corresponde aproximadamente a un año de datos.")
else:
    print("Interpretación: el archivo contiene más de un año de datos.")

In [5]:
def sin_cos_to_angle(s, c):
    ang = np.rad2deg(np.arctan2(s, c))
    return (ang + 360) % 360

def angular_errors(y_true, y_pred):
    diff = np.abs(np.asarray(y_true, dtype=float) - np.asarray(y_pred, dtype=float))
    return np.where(diff > 180, 360 - diff, diff)

def angular_distance(a, b):
    diff = np.abs(np.asarray(a, dtype=float) - np.asarray(b, dtype=float))
    return np.where(diff > 180, 360 - diff, diff)

def rolling_circular_mean_deg(s, window):
    rad = np.deg2rad(s)
    sin_roll = pd.Series(np.sin(rad), index=s.index).rolling(
        window=window,
        center=True,
        min_periods=max(5, window // 3)
    ).mean()
    cos_roll = pd.Series(np.cos(rad), index=s.index).rolling(
        window=window,
        center=True,
        min_periods=max(5, window // 3)
    ).mean()
    return pd.Series(sin_cos_to_angle(sin_roll, cos_roll), index=s.index)

def detectar_atipicos_circulares_por_estacion(df_in, var):
    '''
    Detecta atípicos de dirección del viento respetando su naturaleza circular.
    '''
    banderas = pd.Series(False, index=df_in.index)

    for estacion, sub in df_in.groupby("Estacion"):
        sub = sub.sort_values("Fecha inicial")
        idx = sub.index
        s = pd.to_numeric(sub[var], errors="coerce")

        # Rango válido: [0, 360)
        fuera_rango = s.notna() & ((s < 0) | (s >= 360))

        # Para el análisis circular local se usan únicamente valores válidos.
        s_valid_range = s.where((s >= 0) & (s < 360), np.nan)

        media_circular = rolling_circular_mean_deg(s_valid_range, VENTANA_ANGULAR_ATIPICOS)
        desv_angular = pd.Series(angular_distance(s_valid_range, media_circular), index=s.index)

        desv_valid = desv_angular.dropna()
        if len(desv_valid) >= 8:
            q1 = desv_valid.quantile(0.25)
            q3 = desv_valid.quantile(0.75)
            iqr = q3 - q1
            limite_desv = max(UMBRAL_DESVIACION_ANGULAR, q3 + MULTIPLICADOR_IQR_ANGULAR * iqr)
            atipico_local = desv_angular > limite_desv
        else:
            atipico_local = pd.Series(False, index=s.index)

        banderas.loc[idx] = (fuera_rango | atipico_local.fillna(False)).values

    return banderas.fillna(False)

def build_full(df_in, var):
    estaciones = sorted(df_in["Estacion"].dropna().unique())
    fechas = pd.date_range(df_in["Fecha inicial"].min(), df_in["Fecha inicial"].max(), freq="D")
    grid = pd.MultiIndex.from_product([estaciones, fechas], names=["Estacion", "Fecha inicial"]).to_frame(index=False)

    out = grid.merge(df_in[["Estacion", "Fecha inicial", var]], on=["Estacion", "Fecha inicial"], how="left")
    out[f"{var}_original"] = out[var]
    out["original_missing"] = out[var].isna()
    out["Fue_atipico"] = detectar_atipicos_circulares_por_estacion(out, var)
    out["Requiere_imputacion"] = out["original_missing"] | out["Fue_atipico"]

    # Los atípicos se convierten temporalmente en NaN para ser imputados.
    out.loc[out["Fue_atipico"], var] = np.nan

    out["target_sin"] = np.where(out[var].notna(), np.sin(np.deg2rad(out[var])), np.nan)
    out["target_cos"] = np.where(out[var].notna(), np.cos(np.deg2rad(out[var])), np.nan)
    return out

def add_features(base, var):
    df_feat = base.copy().sort_values(["Estacion", "Fecha inicial"]).reset_index(drop=True)

    fecha = df_feat["Fecha inicial"]
    df_feat["year"] = fecha.dt.year
    df_feat["month"] = fecha.dt.month
    df_feat["day"] = fecha.dt.day
    df_feat["dayofweek"] = fecha.dt.dayofweek
    df_feat["weekofyear"] = fecha.dt.isocalendar().week.astype(int)
    df_feat["dayofyear"] = fecha.dt.dayofyear

    # codificación cíclica del tiempo
    df_feat["sin_doy"] = np.sin(2 * np.pi * df_feat["dayofyear"] / 365.25)
    df_feat["cos_doy"] = np.cos(2 * np.pi * df_feat["dayofyear"] / 365.25)
    df_feat["sin_dow"] = np.sin(2 * np.pi * df_feat["dayofweek"] / 7)
    df_feat["cos_dow"] = np.cos(2 * np.pi * df_feat["dayofweek"] / 7)
    df_feat["sin_month"] = np.sin(2 * np.pi * df_feat["month"] / 12)
    df_feat["cos_month"] = np.cos(2 * np.pi * df_feat["month"] / 12)

    # información de otras estaciones el mismo día
    resumen_fecha = df_feat.groupby("Fecha inicial")[["target_sin", "target_cos"]].agg(["sum", "count"])
    resumen_fecha.columns = ["sum_sin", "count_sin", "sum_cos", "count_cos"]
    df_feat = df_feat.merge(resumen_fecha, left_on="Fecha inicial", right_index=True, how="left")

    self_sin_obs = df_feat["target_sin"].notna().astype(int)
    self_cos_obs = df_feat["target_cos"].notna().astype(int)

    sin_val = df_feat["target_sin"].fillna(0)
    cos_val = df_feat["target_cos"].fillna(0)

    denom_s = df_feat["count_sin"] - self_sin_obs
    denom_c = df_feat["count_cos"] - self_cos_obs

    df_feat["other_station_sin"] = np.where(denom_s > 0, (df_feat["sum_sin"] - sin_val) / denom_s, np.nan)
    df_feat["other_station_cos"] = np.where(denom_c > 0, (df_feat["sum_cos"] - cos_val) / denom_c, np.nan)

    df_feat.drop(columns=["sum_sin", "count_sin", "sum_cos", "count_cos"], inplace=True)

    # rezagos y medias móviles
    lags = [1, 2, 3, 7, 14]
    ventanas = [3, 7, 14]

    for estacion, sub in df_feat.groupby("Estacion"):
        sub = sub.sort_values("Fecha inicial")

        for lag in lags:
            df_feat.loc[sub.index, f"lag_{lag}_sin"] = sub["target_sin"].shift(lag).values
            df_feat.loc[sub.index, f"lag_{lag}_cos"] = sub["target_cos"].shift(lag).values

        for w in ventanas:
            df_feat.loc[sub.index, f"rollmean_{w}_sin"] = sub["target_sin"].shift(1).rolling(w, min_periods=1).mean().values
            df_feat.loc[sub.index, f"rollmean_{w}_cos"] = sub["target_cos"].shift(1).rolling(w, min_periods=1).mean().values

    return df_feat

def get_feature_cols(df_feat, var):
    excluir = ["Estacion", "Fecha inicial", var, f"{var}_original", "original_missing", "Fue_atipico", "Requiere_imputacion", "target_sin", "target_cos"]
    return [c for c in df_feat.columns if c not in excluir]

def make_model():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("rf", RandomForestRegressor(
            n_estimators=300,
            max_depth=12,
            min_samples_leaf=2,
            random_state=42,
            n_jobs=-1
        ))
    ])

def impute_circular_rf(df_in, var):
    base = build_full(df_in, var)
    feat = add_features(base, var)
    feature_cols = get_feature_cols(feat, var)

    train = feat[feat[var].notna()].copy()

    model_sin = make_model()
    model_cos = make_model()

    model_sin.fit(train[feature_cols], train["target_sin"])
    model_cos.fit(train[feature_cols], train["target_cos"])

    work = base.copy()
    missing_idx = work[work["Requiere_imputacion"]].sort_values(["Estacion", "Fecha inicial"]).index.tolist()

    for idx in missing_idx:
        wf = add_features(work, var)
        X = wf.loc[[idx], feature_cols]

        pred_sin = float(model_sin.predict(X)[0])
        pred_cos = float(model_cos.predict(X)[0])

        norma = math.sqrt(pred_sin**2 + pred_cos**2)
        if norma == 0 or np.isnan(norma):
            pred_sin, pred_cos, norma = 0.0, 1.0, 1.0

        pred_sin /= norma
        pred_cos /= norma

        angulo = float(sin_cos_to_angle(pred_sin, pred_cos))

        work.loc[idx, var] = angulo
        work.loc[idx, "target_sin"] = pred_sin
        work.loc[idx, "target_cos"] = pred_cos

    return work

In [6]:
# Diagnóstico inicial de faltantes y atípicos
resultado_diagnostico = build_full(df, VARIABLE)

resumen_calidad = (
    resultado_diagnostico.groupby("Estacion")
    .agg(
        Registros_validos_para_entrenar=(VARIABLE, "count"),
        Dias_esperados=("Fecha inicial", "count"),
        Dias_faltantes=("original_missing", "sum"),
        Atipicos_detectados=("Fue_atipico", "sum"),
        Total_a_imputar=("Requiere_imputacion", "sum")
    )
    .reset_index()
)

resumen_calidad["Porcentaje_total_a_imputar"] = (
    resumen_calidad["Total_a_imputar"] / resumen_calidad["Dias_esperados"] * 100
).round(4)

display(resumen_calidad)

tabla_faltantes_atipicos = resultado_diagnostico.loc[
    resultado_diagnostico["Requiere_imputacion"],
    ["Estacion", "Fecha inicial", f"{VARIABLE}_original", "original_missing", "Fue_atipico"]
].copy()

display(tabla_faltantes_atipicos)

resumen_calidad.to_csv(f"{ARCHIVO_BASE}_resumen_faltantes_atipicos.csv", index=False, encoding="utf-8-sig")
tabla_faltantes_atipicos.to_csv(f"{ARCHIVO_BASE}_fechas_faltantes_y_atipicos.csv", index=False, encoding="utf-8-sig")

,Estacion,Registros_observados,Dias_esperados,Dias_faltantes,Porcentaje_faltante
0,CENTRO DE ALTO RENDIMIENTO,731,731,0,0.0
1,SAN CRISTOBAL,731,731,0,0.0


In [7]:
# Validación automática adaptable a cualquier rango temporal

ken_full = (
    build_full(df, VARIABLE)
    .query("Estacion == 'KENNEDY'")
    .sort_values("Fecha inicial")
    .reset_index(drop=True)
)

# Crear bloques automáticamente
bloques_validacion = []

total_filas = len(ken_full)

# Generar posiciones válidas automáticamente
for inicio in range(30, total_filas - 10, max(30, total_filas // 6)):

    tam = 3 if inicio % 2 == 0 else 4

    if inicio + tam < total_filas:
        bloques_validacion.append((tam, inicio))

resultados_validacion = []

for tam_bloque, inicio in bloques_validacion:

    fechas_bloque = ken_full.loc[
        inicio:inicio + tam_bloque - 1,
        "Fecha inicial"
    ]

    valores_bloque = ken_full.loc[
        inicio:inicio + tam_bloque - 1,
        VARIABLE
    ]

    # Saltar bloques con NaN reales
    if valores_bloque.isna().any():
        continue

    tmp = df.copy()

    mask = (
        (tmp["Estacion"] == "KENNEDY") &
        (tmp["Fecha inicial"].isin(fechas_bloque))
    )

    verdaderos = tmp.loc[mask, VARIABLE].tolist()

    # Crear faltantes artificiales
    tmp = tmp.loc[~mask].copy()

    imputado_tmp = impute_circular_rf(tmp, VARIABLE)

    predichos = (
        imputado_tmp[
            (imputado_tmp["Estacion"] == "KENNEDY") &
            (imputado_tmp["Fecha inicial"].isin(fechas_bloque))
        ]
        .sort_values("Fecha inicial")[VARIABLE]
        .tolist()
    )

    # Verificar tamaños consistentes
    if len(verdaderos) != len(predichos):
        continue

    errores = angular_errors(verdaderos, predichos)

    resultados_validacion.append({
        "inicio": str(fechas_bloque.min().date()),
        "fin": str(fechas_bloque.max().date()),
        "tam_bloque": tam_bloque,
        "MAE_angular": float(np.mean(errores)),
        "RMSE_angular": float(
            np.sqrt(np.mean(np.square(errores)))
        )
    })

tabla_validacion = pd.DataFrame(resultados_validacion)

display(tabla_validacion)

if not tabla_validacion.empty:

    print(
        "MAE angular promedio:",
        round(tabla_validacion["MAE_angular"].mean(), 4)
    )

    print(
        "RMSE angular promedio:",
        round(tabla_validacion["RMSE_angular"].mean(), 4)
    )
else:
    print("No se pudieron generar bloques válidos.")

""


No se pudieron generar bloques válidos.


In [8]:
# Imputación final de faltantes y atípicos
resultado_imputado = impute_circular_rf(df, VARIABLE)

tabla_imputados = resultado_imputado.loc[
    resultado_imputado["Requiere_imputacion"],
    ["Estacion", "Fecha inicial", f"{VARIABLE}_original", VARIABLE, "original_missing", "Fue_atipico"]
].copy()

tabla_imputados = tabla_imputados.rename(columns={VARIABLE: f"{VARIABLE}_final"})
display(tabla_imputados)

,Estacion,Fecha inicial,DViento


In [9]:
# Construcción de datasets finales para exportación
resultado_base = build_full(df, VARIABLE)

resultado_final = resultado_base.merge(
    resultado_imputado[["Estacion", "Fecha inicial", VARIABLE]].rename(columns={VARIABLE: f"{VARIABLE}_final"}),
    on=["Estacion", "Fecha inicial"],
    how="left"
)

resultado_final[f"{VARIABLE}_imputado"] = np.where(
    resultado_final["Requiere_imputacion"],
    resultado_final[f"{VARIABLE}_final"],
    np.nan
)

resultado_final["Fue_faltante_original"] = resultado_final["original_missing"]
resultado_final["Fuente_valor"] = np.select(
    [
        resultado_final["Fue_faltante_original"],
        resultado_final["Fue_atipico"]
    ],
    [
        "Imputado_RF_faltante_original",
        "Imputado_RF_atipico"
    ],
    default="Original"
)

resultado_final["Fecha final"] = ""

resultado_exportar_detallado = resultado_final[
    [
        "Estacion",
        "Fecha inicial",
        "Fecha final",
        f"{VARIABLE}_original",
        f"{VARIABLE}_imputado",
        f"{VARIABLE}_final",
        "Fue_faltante_original",
        "Fue_atipico",
        "Requiere_imputacion",
        "Fuente_valor",
    ]
].copy()

resultado_exportar_detallado = (
    resultado_exportar_detallado
    .sort_values(["Estacion", "Fecha inicial"])
    .reset_index(drop=True)
)

resultado_exportar_final = resultado_exportar_detallado[
    ["Estacion", "Fecha inicial", "Fecha final", f"{VARIABLE}_final", "Fue_faltante_original", "Fue_atipico", "Requiere_imputacion", "Fuente_valor"]
].rename(columns={f"{VARIABLE}_final": VARIABLE})

tabla_imputados_exportar = resultado_exportar_detallado.loc[
    resultado_exportar_detallado["Requiere_imputacion"],
    [
        "Estacion",
        "Fecha inicial",
        "Fecha final",
        f"{VARIABLE}_original",
        f"{VARIABLE}_imputado",
        f"{VARIABLE}_final",
        "Fue_faltante_original",
        "Fue_atipico",
        "Fuente_valor",
    ],
].copy()

display(resultado_exportar_detallado.head())
display(tabla_imputados_exportar)

,Estacion,Fecha inicial,Fecha final,DViento_original,DViento_imputado,DViento_final,Fue_faltante,Fuente_valor
0,CENTRO DE ALTO RENDIMIENTO,2020-01-01,,251.333333,NaN,251.333333,False,Original
1,CENTRO DE ALTO RENDIMIENTO,2020-01-02,,273.625000,NaN,273.625000,False,Original
2,CENTRO DE ALTO RENDIMIENTO,2020-01-03,,255.875000,NaN,255.875000,False,Original
3,CENTRO DE ALTO RENDIMIENTO,2020-01-04,,298.958333,NaN,298.958333,False,Original
4,CENTRO DE ALTO RENDIMIENTO,2020-01-05,,260.500000,NaN,260.500000,False,Original


,Estacion,Fecha inicial,Fecha final,DViento_original,DViento_imputado,DViento_final,Fuente_valor


In [ ]:
# ==========================================
# GRÁFICA PROFESIONAL AUTOMÁTICA: DViento ANTES VS DESPUÉS DE IMPUTAR
# ==========================================

import matplotlib.pyplot as plt
from pathlib import Path

def normalizar_estacion_txt(x):
    return str(x).strip().upper()

def preparar_carpeta_graficas():
    carpeta = Path("graficas_imputacion_antes_despues")
    carpeta.mkdir(parents=True, exist_ok=True)
    return carpeta

def diagnosticar_estaciones(resultado_final):
    estaciones_disp = sorted(
        resultado_final["Estacion"]
        .dropna()
        .astype(str)
        .str.strip()
        .str.upper()
        .unique()
    )
    print("Estaciones disponibles en resultado_final:")
    for e in estaciones_disp:
        print("-", e)
    return estaciones_disp

def graficar_dviento_antes_despues(resultado_final, estacion=None, guardar=True):
    df_plot = resultado_final.copy()
    df_plot["Estacion_norm"] = df_plot["Estacion"].astype(str).str.strip().str.upper()

    estaciones_disp = sorted(df_plot["Estacion_norm"].dropna().unique())

    if not estaciones_disp:
        print("No hay estaciones disponibles para graficar.")
        return None

    if estacion is None:
        estacion = estaciones_disp[0]
        print(f"No se indicó estación. Se grafica automáticamente: {estacion}")

    estacion_norm = normalizar_estacion_txt(estacion)

    if estacion_norm not in estaciones_disp:
        print(f"La estación '{estacion}' no aparece en el dataset.")
        print("Usa una de estas estaciones:")
        for e in estaciones_disp:
            print("-", e)
        return None

    df_estacion = df_plot[df_plot["Estacion_norm"] == estacion_norm].copy()
    df_estacion = df_estacion.sort_values("Fecha inicial")

    col_original = f"{VARIABLE}_original"
    col_final = f"{VARIABLE}_final"

    if col_original not in df_estacion.columns or col_final not in df_estacion.columns:
        print("No se encontraron las columnas esperadas para graficar.")
        print(df_estacion.columns.tolist())
        return None

    fig, ax = plt.subplots(figsize=(15, 6))

    ax.plot(
        df_estacion["Fecha inicial"],
        df_estacion[col_original],
        linestyle="--",
        marker="o",
        markersize=3,
        linewidth=1,
        alpha=0.65,
        label="Antes de imputar / original"
    )

    ax.plot(
        df_estacion["Fecha inicial"],
        df_estacion[col_final],
        linestyle="-",
        linewidth=2,
        label="Después de imputar / DViento final"
    )

    faltantes = (
        df_estacion[df_estacion["Fue_faltante_original"] == True]
        if "Fue_faltante_original" in df_estacion.columns else pd.DataFrame()
    )

    atipicos = (
        df_estacion[df_estacion["Fue_atipico"] == True]
        if "Fue_atipico" in df_estacion.columns else pd.DataFrame()
    )

    if len(faltantes) > 0:
        ax.scatter(faltantes["Fecha inicial"], faltantes[col_final], s=80, marker="o", label="Faltantes imputados", zorder=4)

    if len(atipicos) > 0:
        ax.scatter(atipicos["Fecha inicial"], atipicos[col_final], s=100, marker="x", label="Atípicos corregidos/imputados", zorder=5)

    ax.set_title(f"DViento - {estacion_norm}\nComparación antes y después de imputación circular")
    ax.set_xlabel("Fecha")
    ax.set_ylabel("DViento (grados)")
    ax.set_ylim(-5, 365)
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.tight_layout()

    if guardar:
        carpeta = preparar_carpeta_graficas()
        nombre_seguro = "".join(ch if ch.isalnum() or ch in ["_", "-"] else "_" for ch in estacion_norm)
        ruta_grafica = carpeta / f"DViento_antes_despues_imputacion_{nombre_seguro}.png"
        plt.savefig(ruta_grafica, dpi=150, bbox_inches="tight")
        print("Gráfica guardada en:", ruta_grafica)

    plt.show()

    resumen = pd.DataFrame([{
        "Estacion": estacion_norm,
        "Registros_graficados": len(df_estacion),
        "Valores_originales_no_nulos": int(df_estacion[col_original].notna().sum()),
        "Valores_finales_no_nulos": int(df_estacion[col_final].notna().sum()),
        "Faltantes_originales_imputados": int(faltantes.shape[0]) if len(faltantes) > 0 else 0,
        "Atipicos_corregidos": int(atipicos.shape[0]) if len(atipicos) > 0 else 0
    }])

    display(resumen)
    return resumen

def graficar_todas_las_estaciones_dviento(resultado_final):
    estaciones = diagnosticar_estaciones(resultado_final)
    resumenes = []

    for estacion in estaciones:
        print("\n" + "=" * 80)
        print(f"Generando gráfica para: {estacion}")
        print("=" * 80)
        resumen = graficar_dviento_antes_despues(resultado_final, estacion=estacion, guardar=True)
        if resumen is not None:
            resumenes.append(resumen)

    if resumenes:
        resumen_total = pd.concat(resumenes, ignore_index=True)
        display(Markdown("## Resumen de gráficas generadas"))
        display(resumen_total)
        resumen_total.to_csv("DViento_resumen_graficas_antes_despues.csv", index=False, encoding="utf-8-sig")

graficar_todas_las_estaciones_dviento(resultado_final)

In [10]:
# Exportación de archivos
resultado_exportar_detallado.to_csv(f"{ARCHIVO_BASE}_dataset_completo_imputado_detallado_con_atipicos.csv", index=False, encoding="utf-8-sig")
resultado_exportar_final.to_csv(f"{ARCHIVO_BASE}_dataset_completo_final_con_atipicos.csv", index=False, encoding="utf-8-sig")
tabla_imputados_exportar.to_csv(f"{ARCHIVO_BASE}_valores_faltantes_y_atipicos_imputados.csv", index=False, encoding="utf-8-sig")

with pd.ExcelWriter(f"{ARCHIVO_BASE}_dataset_completo_imputado_detallado_con_atipicos.xlsx", engine="openpyxl") as writer:
    resultado_exportar_detallado.to_excel(writer, sheet_name="detalle_completo", index=False)
    resultado_exportar_final.to_excel(writer, sheet_name="dataset_final", index=False)
    tabla_imputados_exportar.to_excel(writer, sheet_name="solo_imputados", index=False)
    resumen_calidad.to_excel(writer, sheet_name="resumen", index=False)

print("Archivos exportados correctamente:")
print(f"- {ARCHIVO_BASE}_dataset_completo_imputado_detallado_con_atipicos.csv")
print(f"- {ARCHIVO_BASE}_dataset_completo_final_con_atipicos.csv")
print(f"- {ARCHIVO_BASE}_valores_faltantes_y_atipicos_imputados.csv")
print(f"- {ARCHIVO_BASE}_dataset_completo_imputado_detallado_con_atipicos.xlsx")

Archivos exportados correctamente:
- DViento_dataset_completo_imputado_detallado.csv
- DViento_dataset_completo_final.csv
- DViento_valores_imputados.csv
- DViento_dataset_completo_imputado_detallado.xlsx


## Interpretación metodológica final

- La imputación se realizó con **Random Forest**.
- Como `DViento` es una variable angular, se modeló usando **seno** y **coseno**.
- Los valores faltantes y los valores atípicos fueron tratados como registros que requieren imputación.
- Los atípicos se identificaron por rango angular inválido y desviación circular anormal.
- Después de predecir seno y coseno, se reconstruyó el ángulo final en grados.

### Archivos exportados
1. **dataset detallado**: original, imputado, final y trazabilidad.
2. **dataset final**: listo para análisis e integración.
3. **solo imputados**: incluye faltantes originales y atípicos corregidos.
4. **archivo Excel** con detalle, dataset final, solo imputados y resumen.

In [ ]:
# ============================================================
# TIEMPO TOTAL DE EJECUCIÓN DEL NOTEBOOK
# ============================================================

TIEMPO_FIN_NOTEBOOK = time.perf_counter()
FECHA_HORA_FIN_NOTEBOOK = datetime.now()

TIEMPO_TOTAL_SEGUNDOS = TIEMPO_FIN_NOTEBOOK - TIEMPO_INICIO_NOTEBOOK
TIEMPO_TOTAL_MINUTOS = TIEMPO_TOTAL_SEGUNDOS / 60
TIEMPO_TOTAL_HORAS = TIEMPO_TOTAL_MINUTOS / 60

nombre_notebook = globals().get("ARCHIVO_BASE", "notebook_imputacion")
nombre_notebook = str(nombre_notebook).replace(" ", "_").replace(".", "_")

tiempo_ejecucion_df = pd.DataFrame([{
    "Notebook": nombre_notebook,
    "Fecha_hora_inicio": FECHA_HORA_INICIO_NOTEBOOK,
    "Fecha_hora_fin": FECHA_HORA_FIN_NOTEBOOK,
    "Tiempo_total_segundos": round(TIEMPO_TOTAL_SEGUNDOS, 4),
    "Tiempo_total_minutos": round(TIEMPO_TOTAL_MINUTOS, 4),
    "Tiempo_total_horas": round(TIEMPO_TOTAL_HORAS, 4)
}])

display(tiempo_ejecucion_df)

archivo_tiempo_jupyter = f"{nombre_notebook}_TIEMPO_EJECUCION_NOTEBOOK.csv"
tiempo_ejecucion_df.to_csv(archivo_tiempo_jupyter, index=False, encoding="utf-8-sig")

# Guardar también en carpeta de salidas si existe.
try:
    if "TABLAS" in globals():
        tiempo_ejecucion_df.to_csv(TABLAS / f"{nombre_notebook}_TIEMPO_EJECUCION_NOTEBOOK.csv", index=False, encoding="utf-8-sig")
    elif "SALIDAS" in globals():
        tiempo_ejecucion_df.to_csv(SALIDAS / f"{nombre_notebook}_TIEMPO_EJECUCION_NOTEBOOK.csv", index=False, encoding="utf-8-sig")
except Exception as e:
    print("No fue posible guardar copia en carpeta de salidas:", e)

print("Tiempo total de ejecución del notebook:")
print(f"{TIEMPO_TOTAL_SEGUNDOS:.2f} segundos")
print(f"{TIEMPO_TOTAL_MINUTOS:.2f} minutos")
print(f"{TIEMPO_TOTAL_HORAS:.4f} horas")
print("Archivo de tiempo guardado en la carpeta principal del Jupyter:")
print(archivo_tiempo_jupyter)